# HMC Benchmark Notebook

Set `SUMMARY_JSON` below to the benchmark JSON produced by `production_hmc.py benchmark`.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

SUMMARY_JSON = Path("production_benchmark.json")
summary = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
cases = summary["cases"]
labels = [case["name"] for case in cases]
labels


In [ ]:
accept = [case["acceptance_mean"] for case in cases]
ess_local = [case["perf"]["local"]["ess_per_sec_doubleOcc"] for case in cases]
ess_hmc = [case["perf"]["hmc"]["ess_per_sec_doubleOcc"] for case in cases]
tau_local = [case["perf"]["local"]["tau_int_doubleOcc"] for case in cases]
tau_hmc = [case["perf"]["hmc"]["tau_int_doubleOcc"] for case in cases]

x = np.arange(len(labels))
fig, axes = plt.subplots(3, 1, figsize=(12, 12), constrained_layout=True)

axes[0].bar(x, accept, color="#3b6fb6")
axes[0].axhspan(0.70, 0.85, color="#d7ebff")
axes[0].set_ylabel("Acceptance")
axes[0].set_xticks(x, labels, rotation=30, ha="right")

width = 0.38
axes[1].bar(x - width/2, ess_local, width, label="Local", color="#999999")
axes[1].bar(x + width/2, ess_hmc, width, label="HMC", color="#d55e00")
axes[1].set_ylabel("ESS / sec")
axes[1].legend()
axes[1].set_xticks(x, labels, rotation=30, ha="right")

axes[2].bar(x - width/2, tau_local, width, label="Local", color="#999999")
axes[2].bar(x + width/2, tau_hmc, width, label="HMC", color="#009e73")
axes[2].set_ylabel("tau_int(doubleOcc)")
axes[2].legend()
axes[2].set_xticks(x, labels, rotation=30, ha="right")
plt.show()


In [ ]:
focus_obs = ["kinetic", "doubleOcc", "SF_Gamma", "SF_K", "PF_Gamma", "denden_Gamma"]
z = np.zeros((len(focus_obs), len(cases)))
for j, case in enumerate(cases):
    by_name = {row["observable"]: row for row in case["observables"]}
    for i, obs in enumerate(focus_obs):
        z[i, j] = by_name[obs]["z_score"]

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(z, aspect="auto", cmap="magma")
ax.set_xticks(np.arange(len(labels)), labels, rotation=30, ha="right")
ax.set_yticks(np.arange(len(focus_obs)), focus_obs)
ax.set_title("Local vs HMC z-scores")
fig.colorbar(im, ax=ax)
plt.show()
